In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
import shutil
import re

In [3]:
from mutagen import File
import soundfile as sf

In [4]:
import librosa.display
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
def organize_audio_files(audio_folder, keys_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    id_to_key = {}
    
    for filename in os.listdir(keys_folder):
        if filename.endswith('.key'):
            file_path = os.path.join(keys_folder, filename)
            
            file_id = filename.split('.')[0]
            
            with open(file_path, 'r') as f:
                key_data = f.read().strip()
            
            id_to_key[file_id] = key_data
                
    file_count = 0
    id_pattern = re.compile(r'^(\d+)\s')
    
    for filename in os.listdir(audio_folder):
        if filename.endswith('.mp3') or filename.endswith('.wav'):
            match = id_pattern.match(filename)
            if match:
                file_id = match.group(1)
                
                if file_id in id_to_key:
                    key = id_to_key[file_id]
                    
                    key_folder = os.path.join(output_folder, key)
                    if not os.path.exists(key_folder):
                        os.makedirs(key_folder)
                    
                    source_path = os.path.join(audio_folder, filename)
                    dest_path = os.path.join(key_folder, filename)
                    shutil.copy2(source_path, dest_path)  
                    
                    file_count += 1
                else:
                    print(f"Warning: No key data found for {filename}")

In [ ]:
a = '../Data/audio'
k = '../Data/key'
o = '../Data/processed'

In [7]:
# organize_audio_files(a, k, o)

In [5]:
# ---------- #

In [6]:
minor_keys = ['A minor', 'Ab minor', 'B minor', 'Bb minor', 'C minor', 'D minor', 'Db minor', 'E minor', 'Eb minor', 'F minor', 'G minor', 'Gb minor']

In [ ]:
root_folder = '../Data/processed'

audio_files = []
for k in minor_keys:
    key_folder_path = f"{root_folder}/{k}"
    for file in os.listdir(key_folder_path):
        if file.endswith(('.wav', '.mp3', '.ogg')):
            audio_files.append(f"{key_folder_path}/{file}")

durations = []
for file_path in audio_files:
    audio = File(file_path)
    if audio is not None and audio.info is not None:
        durations.append(audio.info.length * 1000) 

min_duration = min(durations)
print(f"Global shortest duration: {min_duration:.2f} ms")

Global shortest duration: 120006.53 ms


In [8]:
#Good all audio clips are around 2mins

In [28]:
# def create_spectrogram(audio_file, image_file):
#     fig = plt.figure()
#     ax = fig.add_subplot(1, 1, 1)
#     fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

#     y, sr = librosa.load(audio_file)
#     ms = librosa.feature.melspectrogram(y=y, sr=sr)
#     log_ms = librosa.power_to_db(ms, ref=np.max)
#     librosa.display.specshow(log_ms, sr=sr)

#     fig.savefig(image_file)
#     plt.close(fig)


# def create_spectrogram(audio_file, image_file, threshold_db=-100, dynamic_range=75):

#     y, sr = librosa.load(audio_file)
    
#     ms = librosa.feature.melspectrogram(y=y, sr=sr)
    
#     log_ms = librosa.power_to_db(ms, ref=np.max)
    
#     log_ms_thresholded = np.maximum(log_ms, np.max(log_ms) + threshold_db)
    
#     fig = plt.figure(figsize=(10, 4))
#     ax = fig.add_subplot(1, 1, 1)
#     fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

#     img = librosa.display.specshow(
#         log_ms_thresholded, 
#         sr=sr, 
#         ax=ax,
#         vmin=np.max(log_ms_thresholded) - dynamic_range 
#     )
    
#     fig.savefig(image_file, dpi=300, bbox_inches='tight', pad_inches=0)
#     plt.close(fig)





# def create_spectrogram(audio_file, image_file, threshold_db=-100, dynamic_range=75, top_n_freqs=5):
#     # Load audio
#     y, sr = librosa.load(audio_file)
    
#     # Generate mel spectrogram
#     ms = librosa.feature.melspectrogram(y=y, sr=sr)
    
#     # Convert to dB
#     log_ms = librosa.power_to_db(ms, ref=np.max)
    
#     # Apply threshold to suppress very low energies
#     log_ms_thresholded = np.maximum(log_ms, np.max(log_ms) + threshold_db)
    
#     # Keep only the top_n_freqs per time frame (column-wise)
#     mask = np.zeros_like(log_ms_thresholded, dtype=bool)
#     for t in range(log_ms_thresholded.shape[1]):
#         # Find the indices of the top n frequencies at this time
#         top_indices = np.argsort(log_ms_thresholded[:, t])[-top_n_freqs:]
#         mask[top_indices, t] = True
    
#     # Zero out non-top frequencies (or set to minimum dB)
#     dominant_ms = np.full_like(log_ms_thresholded, np.max(log_ms_thresholded) + threshold_db)
#     dominant_ms[mask] = log_ms_thresholded[mask]
    
#     # Plotting
#     fig = plt.figure(figsize=(10, 4))
#     ax = fig.add_subplot(1, 1, 1)
#     fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

#     img = librosa.display.specshow(
#         dominant_ms, 
#         sr=sr, 
#         ax=ax,
#         vmin=np.max(dominant_ms) - dynamic_range,
#         cmap='magma'  # Optional: better visual for sparsity
#     )
    
#     fig.savefig(image_file, dpi=300, bbox_inches='tight', pad_inches=0)
#     plt.close(fig)

# def create_chunked_spectrograms(audio_file, output_dir, chunk_duration=10.0, 
#                                 threshold_db=-100, dynamic_range=75, top_n_freqs=5):
#     y, sr = librosa.load(audio_file, sr=None)
#     total_duration = librosa.get_duration(y=y, sr=sr)
    
#     # Calculate number of chunks
#     n_chunks = int(total_duration // chunk_duration)
    
#     for i in range(n_chunks):
#         start_sample = int(i * chunk_duration * sr)
#         end_sample = int((i + 1) * chunk_duration * sr)
        
#         y_chunk = y[start_sample:end_sample]
        
#         # Create Mel Spectrogram
#         ms = librosa.feature.melspectrogram(y=y_chunk, sr=sr)
#         log_ms = librosa.power_to_db(ms, ref=np.max)
#         log_ms_thresholded = np.maximum(log_ms, np.max(log_ms) + threshold_db)
        
#         # Keep only top frequencies
#         mask = np.zeros_like(log_ms_thresholded, dtype=bool)
#         for t in range(log_ms_thresholded.shape[1]):
#             top_indices = np.argsort(log_ms_thresholded[:, t])[-top_n_freqs:]
#             mask[top_indices, t] = True
        
#         dominant_ms = np.full_like(log_ms_thresholded, np.max(log_ms_thresholded) + threshold_db)
#         dominant_ms[mask] = log_ms_thresholded[mask]
        
#         # Plot and save
#         fig = plt.figure(figsize=(10, 4))
#         ax = fig.add_subplot(1, 1, 1)
#         fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
        
#         librosa.display.specshow(
#             dominant_ms, 
#             sr=sr, 
#             ax=ax,
#             vmin=np.max(dominant_ms) - dynamic_range,
#             cmap='magma'
#         )
        
#         chunk_filename = os.path.join(output_dir, f"{os.path.basename(audio_file).split('.')[0]}_chunk_{i}.png")
#         fig.savefig(chunk_filename, dpi=300, bbox_inches='tight', pad_inches=0)
#         plt.close(fig)




# def create_pngs_from_mp3s(input_path, output_path):
#     if not os.path.exists(output_path):
#         os.makedirs(output_path)

#     dir = os.listdir(input_path)

#     for i, file in enumerate(dir):
#         input_file = os.path.join(input_path, file)
#         output_file = os.path.join(output_path, file.replace('.mp3', '.png'))
#         create_spectrogram(input_file, output_file)



# def create_pngs_from_mp3s(input_path, output_path):
#     if not os.path.exists(output_path):
#         os.makedirs(output_path)

#     dir = os.listdir(input_path)

#     for i, file in enumerate(dir):
#         input_file = os.path.join(input_path, file)
#         output_file = os.path.join(output_path, file.replace('.mp3', '.png'))
#         create_chunked_spectrograms(input_file, output_file)

In [7]:
def create_chunked_spectrograms(audio_file, output_dir, chunk_duration=10.0, 
                                threshold_db=-100, dynamic_range=75, top_n_freqs=5):
    y, sr = librosa.load(audio_file, sr=None)
    total_duration = librosa.get_duration(y=y, sr=sr)
    
    # Make sure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Calculate number of chunks
    n_chunks = int(total_duration // chunk_duration)
    
    for i in range(n_chunks):
        start_sample = int(i * chunk_duration * sr)
        end_sample = int((i + 1) * chunk_duration * sr)
        
        y_chunk = y[start_sample:end_sample]
        
        ms = librosa.feature.melspectrogram(y=y_chunk, sr=sr)
        log_ms = librosa.power_to_db(ms, ref=np.max)
        log_ms_thresholded = np.maximum(log_ms, np.max(log_ms) + threshold_db)
        
        mask = np.zeros_like(log_ms_thresholded, dtype=bool)
        for t in range(log_ms_thresholded.shape[1]):
            top_indices = np.argsort(log_ms_thresholded[:, t])[-top_n_freqs:]
            mask[top_indices, t] = True
        
        dominant_ms = np.full_like(log_ms_thresholded, np.max(log_ms_thresholded) + threshold_db)
        dominant_ms[mask] = log_ms_thresholded[mask]
        
        fig = plt.figure(figsize=(10, 4))
        ax = fig.add_subplot(1, 1, 1)
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
        
        librosa.display.specshow(
            dominant_ms, 
            sr=sr, 
            ax=ax,
            vmin=np.max(dominant_ms) - dynamic_range,
            cmap='magma'
        )
        
        # Save each chunk as a separate PNG
        base_name = os.path.splitext(os.path.basename(audio_file))[0]
        chunk_filename = os.path.join(output_dir, f"{base_name}_chunk_{i}.png")
        fig.savefig(chunk_filename, dpi=300, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

def create_pngs_from_mp3s(input_path, output_path):
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    dir = os.listdir(input_path)

    for file in dir:
        input_file = os.path.join(input_path, file)
        # New: Create an output subfolder per audio file (optional but clean)
        # filename_without_ext = os.path.splitext(file)[0]
        # output_subfolder = os.path.join(output_path, filename_without_ext)
        # os.makedirs(input_file, exist_ok=True)
        
        create_chunked_spectrograms(input_file, output_path)


In [ ]:
# for k in minor_keys:   
#     input_path = f"C:/Users/LShel/OneDrive/Documents/Applied_Machine_Learning/Datasets/Data_2/{k}"
#     output_path = f"C:/Users/LShel/OneDrive/Documents/Applied_Machine_Learning/Datasets/Data_2/Spectrogram/{k}"
#     create_pngs_from_mp3s(input_path, output_path)


for k in minor_keys:   
    input_path = f"../Data/processed/{k}"
    output_path = f"../Data/processed/Chromagram/{k}"
    
    if os.path.exists(output_path) and os.listdir(output_path):
        print(f"Skipping {k}: output already exists.")
        continue

    create_pngs_from_mp3s(input_path, output_path)

Skipping A minor: output already exists.
Skipping Ab minor: output already exists.
Skipping B minor: output already exists.
Skipping Bb minor: output already exists.
Skipping C minor: output already exists.


C:\Users\LShel\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


Skipping Db minor: output already exists.
Skipping E minor: output already exists.
Skipping Eb minor: output already exists.
Skipping F minor: output already exists.


In [9]:
from keras.preprocessing import image

def load_images_from_path(path, label):
    images = []
    labels = []

    for file in os.listdir(path):
        images.append(image.img_to_array(image.load_img(os.path.join(path, file), target_size=(224, 224, 3))))
        labels.append((label))
        
    return images, labels

def show_images(images):
    fig, axes = plt.subplots(1, 8, figsize=(20, 20), subplot_kw={'xticks': [], 'yticks': []})

    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i] / 255)
        
x = []
y = []

In [ ]:
label_counter = 0

for k in minor_keys:
    
    images, labels = load_images_from_path(f"../Data/processed/Chromagram/{k}", label_counter)
    
    x += images
    y += labels

    label_counter = label_counter + 1

In [11]:
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, stratify=y, test_size=0.2, random_state=0)

x_train_norm = np.array(x_train) / 255
x_test_norm = np.array(x_test) / 255

y_train_encoded = to_categorical(y_train)
y_test_encoded = to_categorical(y_test)

In [15]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Flatten, Dense

model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Flatten())
model.add(Dense(1024, activation='relu'))
model.add(Dense(12, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

C:\Users\LShel\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 128)  │        36,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │    18,875,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12)             │        12,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,220,748 (73.32 MB)

 Trainable params: 19,220,748 (73.32 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
hist = model.fit(x_train_norm, y_train_encoded, validation_data=(x_test_norm, y_test_encoded), batch_size=10, epochs=20)

Epoch 1/20
256/406 ━━━━━━━━━━━━━━━━━━━━ 1:30 604ms/step - accuracy: 0.1456 - loss: 2.4581

KeyboardInterrupt: 

In [ ]:
#classify for each chunk in input song and take majority vote